In [1]:
import glob
import re

import numpy as np
from scipy import stats
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math
from utils import Graph
from mpl_toolkits.mplot3d import Axes3D
import itertools

file_tree = {}
for i in range(0, 100):  
    for file in glob.glob(f'simulation-runs/500V_{i}.txt'):
        key = int(re.search(r'(\d+)', file).group(1))
        if key in file_tree:
            file_tree[key].append(file)
        else:
            file_tree[key] = [file]



## Recombination Classification

Note: If the electron says undetermined, usually the electron has traveled far enough away from any ion and so it would technically have escaped. If you see consistently the electrons being labeled as "undetermined", it means to run the simulation for longer. This notably applies for lower E-field magnitudes.

In [4]:
import glob
import re
import numpy as np

angle    = 80      # degrees — MUST match the run
r_k      = 0.03# µm ion spacing — MUST match the run (density_nm / 1000)
X_ESCAPE = 4.9     # µm; C++ escape boundary is |x| >= 5 µm
KE_CAP   = 1.0      # eV; C++ recombination requires ke < 1.0 eV
R_RECOMB = 0.00129    # µm = 1.29 nm 

cosA, sinA = np.cos(np.deg2rad(angle)), np.sin(np.deg2rad(angle))

# Run range (inclusive)
run_start = 1
run_end   = 100

def run_number(path):
    match = re.search(r'_(\d+)\.txt$', path)
    return int(match.group(1)) if match else -1

selected_files = []
for i in range(run_start, run_end + 1):
    matches = glob.glob(f'simulation-runs/500V_{i}.txt')
    if not matches:
        print(f'Warning: missing file for run {i}')
    selected_files.extend(matches)

selected_files = sorted(selected_files, key=run_number)
total_runs_loaded = len(selected_files)
print(f'Loaded {total_runs_loaded} files for runs {run_start}..{run_end}')

def classify(file):
    """Classify from the true terminating (last) row.
    Returns (outcome, x, ke, dist_to_ion_nm)."""
    with open(file, 'rb') as f:
        f.seek(0, 2)
        size = f.tell()
        if size == 0:
            return 0, None, None, None
        f.seek(max(0, size - 4000))
        last = f.read().splitlines()[-1].decode()
    cols = [c for c in last.strip().split(',') if c != '']
    if len(cols) < 5:
        return 0, None, None, None
    x, y, z, ke = float(cols[1]), float(cols[2]), float(cols[3]), float(cols[4])

    # Escape takes priority (matches C++ `escaped || recombined`)
    if abs(x) >= X_ESCAPE:
        return 1, x, ke, None

    # Perpendicular distance to the nearest ion on the tilted lattice
    s_proj = x * cosA + z * sinA
    s_near = round(s_proj / r_k) * r_k
    xi, zi = s_near * cosA, s_near * sinA
    dist = np.sqrt((x - xi)**2 + y**2 + (z - zi)**2)

    if dist <= R_RECOMB and ke < KE_CAP:
        return -1, x, ke, dist * 1e3   # nm
    return 0, x, ke, dist * 1e3        # nm

occurence_list = []
for file in selected_files:
    outcome, x_last, ke_last, d_nm = classify(file)
    occurence_list.append(outcome)
    label = {1: 'escapes', -1: 'recombines', 0: 'undetermined'}[outcome]
    dstr = f'{d_nm:.2f} nm' if d_nm is not None else '   -   '
    print(f'{file}  ->  {label:12s} (x={x_last} um, ke={ke_last} eV, dist_to_ion={dstr})')

if total_runs_loaded == 0:
    raise ValueError('No simulation files were loaded for the selected run range.')

escape_count       = occurence_list.count(1)
recomb_count       = occurence_list.count(-1)
undetermined_count = occurence_list.count(0)

prob_recomb = recomb_count / total_runs_loaded
prob_escape = escape_count / total_runs_loaded

print('\nOccurrence List:', occurence_list)
print(f'Counts -> escape: {escape_count}, recombination: {recomb_count}, '
      f'undetermined: {undetermined_count}, total: {total_runs_loaded}')
print(f'Escape Probability: {prob_escape:.6f}   Recombination Probability: {prob_recomb:.6f}')

Loaded 100 files for runs 1..100
simulation-runs/500V_1.txt  ->  recombines   (x=0.0210791 um, ke=0.962796 eV, dist_to_ion=1.23 nm)
simulation-runs/500V_2.txt  ->  recombines   (x=0.0989946 um, ke=0.996544 eV, dist_to_ion=1.23 nm)
simulation-runs/500V_3.txt  ->  escapes      (x=5.00005 um, ke=0.0381352 eV, dist_to_ion=   -   )
simulation-runs/500V_4.txt  ->  escapes      (x=5.00029 um, ke=0.0618255 eV, dist_to_ion=   -   )
simulation-runs/500V_5.txt  ->  recombines   (x=-0.119275 um, ke=0.995998 eV, dist_to_ion=1.25 nm)
simulation-runs/500V_6.txt  ->  escapes      (x=5.00018 um, ke=0.0265219 eV, dist_to_ion=   -   )
simulation-runs/500V_7.txt  ->  recombines   (x=0.202275 um, ke=0.994854 eV, dist_to_ion=1.21 nm)
simulation-runs/500V_8.txt  ->  recombines   (x=-0.108395 um, ke=0.954006 eV, dist_to_ion=1.25 nm)
simulation-runs/500V_9.txt  ->  recombines   (x=0.0633351 um, ke=0.910437 eV, dist_to_ion=1.25 nm)
simulation-runs/500V_10.txt  ->  recombines   (x=-0.254145 um, ke=0.989269 eV, d

## Geminate Recombination Check

Checks whether or not of the electrons that recombine, how many recombine with their parent ion.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Geminate vs. columnar recombination.
#
# VALID ONLY when every electron spawns at the origin ion (0,0,0), i.e.
# Use_Finite_Ion_Distribution = false in Constants.h. Under that setup every
# electron is born next to the SAME origin ion, so of the electrons that
# recombine:
#   - recombined AT the origin      -> fell back into its own parent ion -> GEMINATE
#   - recombined at a different ion  -> captured by another track ion     -> COLUMNAR
#
# Uses the true terminating frame (last line, written by the C++) and tests its
# distance from (0,0,0). Depends on occurence_list / selected_files from the
# "Recombination Classification" cell above.
# ---------------------------------------------------------------------------

R_GEMINATE = 0.00129   # µm = 1.29 nm; capture radius around the origin ion

def last_position(file):
    with open(file, 'rb') as f:
        f.seek(0, 2); size = f.tell()
        if size == 0:
            return None
        f.seek(max(0, size - 4000))
        last = f.read().splitlines()[-1].decode()
    cols = [c for c in last.strip().split(',') if c != '']
    if len(cols) < 5:
        return None
    return float(cols[1]), float(cols[2]), float(cols[3])   # x, y, z (µm)

geminate = 0
columnar = 0
details  = []
for outcome, file in zip(occurence_list, selected_files):
    if outcome != -1:            # only recombinations
        continue
    pos = last_position(file)
    if pos is None:
        continue
    x, y, z = pos
    d_origin = np.sqrt(x*x + y*y + z*z)      # distance from (0,0,0) in µm
    kind = 'geminate' if d_origin <= R_GEMINATE else 'columnar'
    if kind == 'geminate':
        geminate += 1
    else:
        columnar += 1
    details.append((file, d_origin * 1e3, kind))   # nm

for file, d_nm, kind in details:
    print(f'{file}  ->  {kind:9s} (dist from origin = {d_nm:.3f} nm)')

n_recomb = geminate + columnar
print(f'\nRecombinations analyzed: {n_recomb}')
if n_recomb > 0:
    print(f'  Geminate (<= {R_GEMINATE*1e3:.2f} nm from origin): {geminate}  '
          f'({geminate/n_recomb:.1%} of recombinations)')
    print(f'  Columnar (captured by another ion):       {columnar}  '
          f'({columnar/n_recomb:.1%} of recombinations)')
else:
    print('  No recombinations found in occurence_list.')

## Max travel distance for recombined electrons

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Max distance from the nearest ion reached by each RECOMBINED electron.
#
# For every row (timestep) of a recombined electron's file, find the nearest ion
# on the tilted track (same geometry / angle / r_k as the classification cell)
# and compute
#     r = sqrt((x - x_ion)^2 + (y - y_ion)^2 + (z - z_ion)^2)
# r_max is the largest r over the whole trajectory. Depends on occurence_list /
# selected_files / angle / r_k / cosA / sinA from the cells above.
# ---------------------------------------------------------------------------

N_IONS = None   # total ions on the track (N+1 for even batches N, N for odd); None = infinite track

def max_dist_to_nearest_ion_nm(file):
    data = np.loadtxt(file, delimiter=',', usecols=(1, 2, 3), ndmin=2)   # x, y, z (µm)
    x, y, z = data[:, 0], data[:, 1], data[:, 2]

    # Nearest ion index along the track, clipped to the ends of a finite track
    k = np.round((x * cosA + z * sinA) / r_k)
    if N_IONS is not None:
        k = np.clip(k, -(N_IONS // 2), N_IONS // 2)
    xi, zi = k * r_k * cosA, k * r_k * sinA

    r = np.sqrt((x - xi)**2 + y**2 + (z - zi)**2)
    return r.max() * 1e3   # µm -> nm

r_max_list = []
r_max_files = []
for outcome, file in zip(occurence_list, selected_files):
    if outcome != -1:            # only recombinations
        continue
    r_max_list.append(max_dist_to_nearest_ion_nm(file))
    r_max_files.append(file)

for i, (file, r_max) in enumerate(zip(r_max_files, r_max_list), start=1):
    print(f'r_max{i} = {r_max:.4f} nm   ({file})')

print(f'\nRecombined electrons: {len(r_max_list)}')
print('r_max (nm):', ', '.join(f'{r:.4f}' for r in r_max_list))
